Esperimenti preliminari 

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
import kagglehub

# 1. CORE AIJACK IMPORTS (Basati sulla tua scansione)
from aijack.collaborative.fedavg import FedAVGClient, FedAVGServer

# Difesa: importiamo dai percorsi esatti che abbiamo visto nella scansione
from aijack.defense.dp import DPSGDManager, GeneralMomentAccountant
from aijack.defense.dp.manager.client import attach_dpsgd_to_client

# Attacco: il nome esatto è GradientInversion_Attack
from aijack.attack.inversion import GradientInversion_Attack

# 2. HARDWARE SETUP
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"🚀 Device pronto: {device}")

# 3. DATASET & PREPROCESSING
path = kagglehub.dataset_download("tawsifurrahman/covid19-radiography-database")
data_dir = os.path.join(path, 'COVID-19_Radiography_Dataset')

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)
# Usiamo pochi dati per velocizzare il test
train_loader = DataLoader(Subset(full_dataset, range(10)), batch_size=1, shuffle=True)

# 4. MODEL DEFINITION
def get_medical_model():
    # Carichiamo resnet18
    model = models.resnet18(weights=None)
    
    # TRUCCO PER L'ATTACCO: Sostituiamo il MaxPool con AvgPool
    # Questo risolve il RuntimeError delle derivate di secondo ordine
    model.maxpool = nn.AvgPool2d(kernel_size=3, stride=2, padding=1)
    
    model.fc = nn.Linear(model.fc.in_features, 4)
    return model.to(device)

# 5. DEFENSE LAYER: DP-SGD
noise_multiplier = 0.5 
accountant = GeneralMomentAccountant(noise_type="Gaussian", backend="python")
privacy_manager = DPSGDManager(
    accountant, 
    optim.SGD, 
    l2_norm_clip=1.0, 
    dataset=full_dataset, 
    lot_size=1, 
    batch_size=1, 
    iterations=1
)

# Funzione di "attacco" della DP al client
DPClientClass, _ = attach_dpsgd_to_client(FedAVGClient, privacy_manager, noise_multiplier)

# Inizializziamo il client difeso
client = DPClientClass(get_medical_model(), user_id=0)

# 6. ATTACK SETUP
# Nota: GradientInversion_Attack usa x_shape invece di input_shape in questa versione
medical_spy = GradientInversion_Attack(
    get_medical_model(), 
    x_shape=(3, 64, 64), 
    device=device
)

# 7. ESECUZIONE
print("Simulazione aggiornamento locale con protezione DP...")
inputs, targets = next(iter(train_loader))
inputs, targets = inputs.to(device), targets.to(device)

# Calcolo gradienti
client.model.zero_grad()
outputs = client.model(inputs)
loss = nn.CrossEntropyLoss()(outputs, targets)
loss.backward()

# Estrazione gradienti (AIJack gestisce il rumore internamente o tramite il manager)
target_gradients = [p.grad.clone() for p in client.model.parameters() if p.grad is not None]

print("Avvio attacco di inversione...")
# In questa versione il metodo si chiama group_attack o attack. Proviamo attack:
try:
    reconstructed_data, _ = medical_spy.group_attack(target_gradients, batch_size=1)
except:
    reconstructed_data = medical_spy.attack(target_gradients)

# 8. VISUALIZZAZIONE
def show(img_tensor, title):
    img = img_tensor[0].cpu().detach().permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    plt.imshow(img)
    plt.title(title)
    plt.axis('off')

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
show(inputs, "Originale (Privato)")
plt.subplot(1, 2, 2)
show(reconstructed_data, f"Ricostruito (σ={noise_multiplier})")
plt.show()

🚀 Device pronto: mps
Simulazione aggiornamento locale con protezione DP...
Avvio attacco di inversione...
worker_id=0: iter=10: 47407.73046875, (best_iter=10: 47407.73046875)
worker_id=1: iter=10: 49614.69140625, (best_iter=2: 47527.80859375)
worker_id=2: iter=10: 49225.140625, (best_iter=3: 47523.52734375)
worker_id=3: iter=10: 53702.4140625, (best_iter=2: 47719.2734375)
worker_id=4: iter=10: 48262.28515625, (best_iter=3: 47727.5859375)
